# `examples/request_gen` 请求生成器示例命令行等价运行版

本 Notebook 对应 `examples/request_gen/main.py`，用于演示 faas-sim 的请求生成器如何驱动函数调用实验。

该示例的核心机制是：

1. 创建基础拓扑；
2. 注册 `python-pi-cpu` 函数镜像；
3. 部署 `python-pi` 函数；
4. 等待函数副本进入可调用状态；
5. 使用 `constant_rps_profile(rps=20)` 构造固定平均请求速率；
6. 使用 `expovariate_arrival_profile(...)` 把目标 RPS 转换为指数分布请求到达间隔；
7. 使用 `function_trigger(...)` 按请求到达间隔触发函数调用；
8. 最多生成 100 个函数请求。

当前版本采用“命令行等价运行”方式，不在 Notebook 中重新拆解实现样例逻辑，而是直接执行原始 `.py` 文件：

```bash
python -u examples/request_gen/main.py
```

这样可以最大程度保持与 PowerShell / 终端运行样例时一致，避免 Notebook 直接调用 `sim.run()` 时出现日志不显示、输出缓存或看起来卡住的问题。

## 1. 定位项目根目录

下面的代码会从当前 Notebook 所在目录开始向上查找项目根目录。

判断标准是：目录中同时存在 `sim/` 和 `examples/`。

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

current_dir = Path.cwd().resolve()

candidate_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
    current_dir.parent.parent.parent,
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / "sim").exists() and (root / "examples").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "没有找到 faas-sim 项目根目录。请把 Notebook 放在项目根目录、examples/request_gen/ "
        "或其相邻目录下运行。"
    )

script_path = PROJECT_ROOT / "examples" / "request_gen" / "main.py"

print(f"当前 Notebook 工作目录：{current_dir}", flush=True)
print(f"faas-sim 项目根目录：{PROJECT_ROOT}", flush=True)
print(f"即将运行的样例脚本：{script_path}", flush=True)

if not script_path.exists():
    raise FileNotFoundError(f"找不到样例脚本：{script_path}")

## 2. 使用当前 Jupyter 内核对应的 Python 执行样例

这里使用 `sys.executable`，保证 Notebook 当前内核使用哪个 Python，就用哪个 Python 来运行样例脚本。

`-u` 参数表示 unbuffered，可以让日志和 `print` 尽快刷新到 Notebook 输出区。

In [ ]:
print("当前 Jupyter 内核 Python：", sys.executable, flush=True)
print("开始以命令行等价方式运行 request_gen 样例。", flush=True)

## 3. 执行 `examples/request_gen/main.py`

这一格会实时打印子进程输出。

如果样例正常运行，你应该看到类似下面的日志：

```text
INFO:sim.faassim:initializing simulation...
INFO:examples.request_gen.main:python-pi-cpu, latest, [...]
INFO:sim.faas.system:deploying function python-pi with scale_min=1
INFO:examples.request_gen.main:waiting for replica
INFO:sim.faas.system:pod pod-python-pi-1 was scheduled to ...
INFO:sim.faassim:simulation ran ...
```

由于本示例会通过请求生成器触发 100 个请求，运行时间和日志数量会比 `basic` 示例更多。

In [ ]:
env = os.environ.copy()

# 确保子进程优先从项目根目录导入本地 sim、examples、ether、skippy、simpy 等包。
existing_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = (
    str(PROJECT_ROOT)
    if not existing_pythonpath
    else str(PROJECT_ROOT) + os.pathsep + existing_pythonpath
)

cmd = [
    sys.executable,
    "-u",
    str(script_path),
]

print("执行命令：", " ".join(cmd), flush=True)
print("工作目录：", PROJECT_ROOT, flush=True)
print("开始输出子进程日志：", flush=True)

process = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

# 实时转发子进程输出。
for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

print(f"\n子进程退出码：{return_code}", flush=True)

if return_code != 0:
    raise RuntimeError(f"request_gen 样例运行失败，退出码：{return_code}")
else:
    print("request_gen 样例运行完成。", flush=True)

## 4. 样例代码机制说明

该样例中的请求生成部分主要是下面三行：

```python
ia_generator = expovariate_arrival_profile(constant_rps_profile(rps=20))
yield from function_trigger(env, deployments[0], ia_generator, max_requests=100)
```

含义如下：

| 组件 | 作用 |
|---|---|
| `constant_rps_profile(rps=20)` | 生成固定目标请求速率曲线，表示平均每秒 20 个请求 |
| `expovariate_arrival_profile(...)` | 将目标 RPS 转换为指数分布的请求到达间隔 |
| `function_trigger(...)` | 按到达间隔不断触发函数请求 |
| `max_requests=100` | 最多生成 100 个请求，防止仿真无限运行 |

为什么使用指数分布？

固定 RPS 并不意味着每个请求都严格等间隔到达。实际请求通常具有随机性。指数分布到达间隔常用于描述泊松到达过程，能模拟“平均速率固定，但单个请求到达时间随机”的场景。

## 5. 后续分析建议

当前 Notebook 只负责把原始 `.py` 样例稳定跑起来。

如果后续要做交互式分析，可以再单独创建分析版 Notebook，直接构造 `Simulation` 对象并在运行结束后读取：

```python
invocations_df = sim.env.metrics.extract_dataframe("invocations")
schedule_df = sim.env.metrics.extract_dataframe("schedule")
flow_df = sim.env.metrics.extract_dataframe("flow")
```

其中，`invocations_df` 最适合分析请求生成器效果，例如：

1. 请求数量是否达到 100；
2. 请求到达时间是否符合指数分布；
3. 函数执行时间是否随负载变化；
4. 请求完成时间与发起时间之间的差异；
5. 不同 RPS 参数下平均响应时间是否变化。

当前阶段建议先确保所有 `.py` 样例通过“命令行等价运行版”稳定跑通，再进入交互式指标分析。